# Gold — Star Schema for Semantic Model
Business-friendly `gold_*` dims/facts plus a shared `gold_dim_date`.

In [ ]:
# ============================================================
# NOTEBOOK 3: GOLD  —  star schema for Power BI semantic model
# ============================================================
from pyspark.sql import functions as F

S, G = "silver_", "gold_"
def slv(t): return spark.table(f"{S}{t}")

# ---------- shared Date dimension (spans encounters + claims) ----------
bounds = (slv("fact_claim").select(F.min("ServiceDate").alias("mn"), F.max("ServiceDate").alias("mx"))
    .join(slv("fact_encounter").select(F.min("EncounterDate").alias("emn"),F.max("EncounterDate").alias("emx"))))
row = bounds.collect()[0]
start = min(row["mn"], row["emn"]); end = max(row["mx"], row["emx"])
dim_date = (spark.sql(f"SELECT explode(sequence(to_date('{start}'), to_date('{end}'), interval 1 day)) AS Date")
    .withColumn("DateKey", F.date_format("Date","yyyyMMdd").cast("int"))
    .withColumn("Year", F.year("Date")).withColumn("Quarter", F.quarter("Date"))
    .withColumn("Month", F.month("Date")).withColumn("MonthName", F.date_format("Date","MMMM"))
    .withColumn("YearMonth", F.date_format("Date","yyyy-MM"))
    .withColumn("DayOfWeek", F.date_format("Date","EEEE")))
dim_date.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{G}dim_date")
print(f"{G+'dim_date':30s} {dim_date.count():>8,} rows")

# ---------- conformed dimensions (rename to business-friendly) ----------
gold_dims = {
  "dim_provider": slv("dim_provider").select("ProviderKey","NPI","ProviderFullName","Credentials",
      "Gender","ProviderType","PrimarySpecialty","ProviderStatus","IsActive","HireDate","TerminationDate"),
  "dim_patient": slv("dim_patient").select("PatientKey","MRN","PatientName","DateOfBirth","Age","AgeBand",
      "Gender","Race","Ethnicity","ZIP","PCPProviderKey"),
  "dim_department": slv("dim_department").select("DepartmentKey","DepartmentName","ServiceLine","DepartmentSpecialty"),
  "dim_facility": slv("dim_facility").select("FacilityKey","FacilityName","City","StateAbbr","ZIP","Region"),
  "dim_payer": slv("dim_payer").select("PayerKey","PayerName","PlanType","LineOfBusiness"),
  "dim_diagnosis": slv("dim_diagnosis").select("DiagnosisKey","ICD10Code","DiagnosisName","DiagnosisCategory"),
  "dim_procedure": slv("dim_procedure").select("ProcedureKey","ProcedureCode","CodeType","ProcedureDescription"),
}
for name, df in gold_dims.items():
    df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{G}{name}")
    print(f"{G+name:30s} {df.count():>8,} rows")

# ---------- fact tables (add DateKey pointing at Dim_Date) ----------
fact_encounter = (slv("fact_encounter")
    .withColumn("DateKey", F.date_format("EncounterDate","yyyyMMdd").cast("int"))
    .select("EncounterKey","DateKey","PatientKey","AttendingProviderKey","ReferringProviderKey",
            "DepartmentKey","LocationKey","DiagnosisKey","EncounterType","LengthOfStay"))
fact_claim = (slv("fact_claim")
    .withColumn("DateKey", F.date_format("ServiceDate","yyyyMMdd").cast("int"))
    .select("ClaimKey","DateKey","PatientKey","BillingProviderKey","RenderingProviderKey","PayerKey",
            "ProcedureKey","DiagnosisKey","BilledAmount","AllowedAmount","PaidAmount",
            "PatientResponsibility","AllowedVariance","ClaimStatus","DeniedFlag"))
fact_risk = (slv("fact_risk_score")
    .withColumn("DateKey", F.date_format("ScoreDate","yyyyMMdd").cast("int"))
    .select("RiskScoreKey","DateKey","PatientKey","ProviderKey","RiskModel","RiskScore"))

for name, df in {"fact_encounter":fact_encounter,"fact_claim":fact_claim,"fact_risk_score":fact_risk}.items():
    df.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"{G}{name}")
    print(f"{G+name:30s} {df.count():>8,} rows")
print("GOLD complete.")